In [ ]:
!nvidia-smi

FILE = "multiheadatten/1st.cu"   # change this to the kernel in kernels/ you want to run

!rm -rf /content/repo && git clone -q --depth 1 https://github.com/Aakash-Rajput-2024/cuda-training.git /content/repo

import os, subprocess
src = "/content/repo/kernels/" + FILE
exe = "/tmp/" + os.path.splitext(FILE)[0]
os.makedirs(os.path.dirname(exe), exist_ok=True)

if not os.path.exists(src):
    print("Not found:", FILE)
    print("Available:", os.listdir("/content/repo/kernels"))
else:
    b = subprocess.run(["nvcc", "-O2", "-arch=sm_75", "-lineinfo", src, "-o", exe],
                       capture_output=True, text=True)
    if b.returncode:
        print(b.stderr)
    else:
        print(subprocess.run([exe], capture_output=True, text=True).stdout, end="")

### Benchmark Multi-Head Attention Kernels
Automatically compiles and runs all kernels inside `kernels/multiheadatten/` side-by-side.

In [ ]:
import os, subprocess, re
import pandas as pd

mha_dir = "/content/repo/kernels/multiheadatten"
files = sorted([f for f in os.listdir(mha_dir) if f.endswith(".cu")]) if os.path.exists(mha_dir) else []

results = []
baseline_lat = None

for f in files:
    src = os.path.join(mha_dir, f)
    exe = "/tmp/mha_" + os.path.splitext(f)[0]
    
    b = subprocess.run(["nvcc", "-O2", "-arch=sm_75", "-lineinfo", src, "-o", exe],
                       capture_output=True, text=True)
    if b.returncode:
        print(f"[Error] Compiling {f}:\n", b.stderr)
        continue
    
    run_res = subprocess.run([exe], capture_output=True, text=True)
    out = run_res.stdout
    print(out)
    
    lat_match = re.search(r"Avg Latency\s*:\s*([\d\.]+)\s*ms", out)
    gflop_match = re.search(r"Throughput\s*:\s*([\d\.]+)\s*GFLOP/s", out)
    lat = float(lat_match.group(1)) if lat_match else None
    gflops = float(gflop_match.group(1)) if gflop_match else None
    
    if baseline_lat is None and lat:
        baseline_lat = lat
    speedup = (baseline_lat / lat) if (baseline_lat and lat) else 1.0
    
    results.append({
        "Kernel": f,
        "Status": "PASS" if "PASS" in out else "FAIL",
        "Latency (ms)": lat,
        "Throughput (GFLOP/s)": gflops,
        "Speedup": f"{speedup:.2f}x"
    })

if results:
    df = pd.DataFrame(results)
    print("\n================ Multi-Head Attention Summary ================")
    print(df.to_string(index=False))


### Plot MHA Performance Comparison

In [ ]:
import matplotlib.pyplot as plt

if results and any(r["Latency (ms)"] for r in results):
    names = [r["Kernel"] for r in results if r["Latency (ms)"]]
    lats = [r["Latency (ms)"] for r in results if r["Latency (ms)"]]
    speedups = [float(r["Speedup"].replace("x", "")) for r in results if r["Latency (ms)"]]
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
    
    # Latency
    bars1 = ax1.bar(names, lats, color="#4C72B0", width=0.4)
    ax1.set_title("Latency (ms) - Lower is better")
    ax1.set_ylabel("ms")
    ax1.grid(axis="y", linestyle="--", alpha=0.5)
    for b in bars1:
        ax1.text(b.get_x() + b.get_width()/2, b.get_height(), f"{b.get_height():.3f} ms", ha="center", va="bottom")
    
    # Speedup
    bars2 = ax2.bar(names, speedups, color="#55A868", width=0.4)
    ax2.axhline(1.0, color="gray", linestyle="--", label="1.0x (1st.cu)")
    ax2.set_title("Speedup vs Baseline - Higher is better")
    ax2.set_ylabel("Speedup Multiplier")
    ax2.grid(axis="y", linestyle="--", alpha=0.5)
    ax2.legend()
    for b in bars2:
        ax2.text(b.get_x() + b.get_width()/2, b.get_height(), f"{b.get_height():.2f}x", ha="center", va="bottom")
    
    plt.tight_layout()
    plt.show()
